# Data Exploration

In this script we put various informative plots or derive important figures (like number of full data points, etc). None of these scripts make changes to the data or do complex analysis; just explore base dataset

# GCproperties.csv

This file contains cluster-level info on our sample. Let's load up GCproperties file and see what it looks like:

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from IPython.core.magic import register_cell_magic

@register_cell_magic
def skip(line, cell):
    return


data_folder='../../Datasets/Original_Datasets'
GCprop=pd.read_csv(f'{data_folder}/GCproperties.csv', encoding='latin1')
GCprop.head()

,#Name,mu_alpha,mu_alpha_err,mu_delta,mu_delta_err,V_los,V_los_err,R_apo,R_apo_err,R_peri,...,sigma+err_Kirby,sigma-err_Kirby,FeH_supp,FeH+err_supp,FeH-err_supp,sigma_FeH,sigma_FeH+err,sigma_FeH-err,FeH_Ref,FeH_notes
0,# Units: mas/yr (proper motions),km/s (velocity),kpc (R_apo,R_peri),1e5 Msun (M_i),pc (r_J),NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,# Refs: (1) Dinescu et al. (1997); (2) Dinescu...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,# (6) Casetti-Dinescu et al. (2010); (7) Caset...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"# (12) Willman & Strader (2012), (13) Worley &...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"# (16) Pohl (2015; PhD), (17) Villanova et al....",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


A lot of this file is actually notes on where the data is from,  not the actual data. So we drop all columns with NA for age, this gets rid of all that and also where we actually have NA for age

In [2]:
GC_leftover=GCprop.dropna(subset=['age_Kruijssen'])

# NGC_combi.csv

This file contains our main dataset, before being split into test/train

In [3]:
combined_data='../Datasets/Original_Datasets/NGC_combi.csv'
df_full=pd.read_csv(combined_data)
df_full=df_full.set_index('Star ID') #this is a unique identifier for each star, instead of each cluster
df_full.rename(columns={'F435W': 'F438W', 'F435W_abs':'F438W_abs', 'F435W_e':'F438W_e'}, inplace=True)   #the F438 filter is misnamed in these files for some reason

#### Check for repeat data:

In [4]:
num_mentions=df_full.index.value_counts()
print(num_mentions)
print(f'Some data is repeated {num_mentions.iloc[0]} times! Other data repeated {num_mentions.iloc[1]} times...not sure yet what to do with this. For now, taking all repeate data completely out when doig train/test split')

Star ID
6.08E+18    60
R0000101     4
R0000373     4
R0000404     4
R0001674     4
            ..
R0004771     1
R0004780     1
R0004864     1
R0004884     1
R0003433     1
Name: count, Length: 858, dtype: int64
Some data is repeated 60 times! Other data repeated 4 times...not sure yet what to do with this. For now, taking all repeate data completely out when doig train/test split


Also, the data point that is repeated 60 times has a weird Star ID. 

In [5]:
num_mentions.index

Index(['6.08E+18', 'R0000101', 'R0000373', 'R0000404', 'R0001674', 'R0000781',
       'R0001469', 'R0000346', 'R0000181', 'R0002301',
       ...
       'R0004346', 'R0004479', 'R0004732', 'R0004739', 'R0004737', 'R0004771',
       'R0004780', 'R0004864', 'R0004884', 'R0003433'],
      dtype='object', name='Star ID', length=858)

Print info on the two most-repeated data points:

In [6]:
df_full.loc[num_mentions.index[0]]

,NGC,Paper,Author Reference,RAJ2000,DEJ2000,Distance (pc),neg_sigma_dist,pos_sigma_dist,Vmag,Vmag_abs,...,F336W_e,F438W,F438W_abs,F438W_e,F606W,F606W_abs,F606W_e,F814W,F814W_abs,F814W_e
Star ID,,,,,,,,,,,,,,,,,,,,,
6.08E+18,5139,Mucciarelli,150704,201.708583,-47.513114,5426,-47,47,14.885,1.212601,...,0.0071,15.8378,2.165401,0.0094,14.7228,1.050401,NaN,13.9047,0.232301,0.0075
6.08E+18,5139,Mucciarelli,150996,201.736107,-47.512880,5426,-47,47,14.836,1.163601,...,0.0079,15.7425,2.070101,0.0094,14.5959,0.923501,NaN,13.7791,0.106701,0.0075
6.08E+18,5139,Mucciarelli,158663,201.711761,-47.506769,5426,-47,47,14.671,0.998601,...,0.0109,15.4938,1.821401,0.0088,14.4352,0.762801,NaN,13.6885,0.016101,0.0075
6.08E+18,5139,Mucciarelli,159324,201.729075,-47.506264,5426,-47,47,15.341,1.668601,...,0.0059,16.4917,2.819301,0.0109,15.0914,1.419001,NaN,14.2593,0.586901,0.0075
6.08E+18,5139,Mucciarelli,159562,201.672705,-47.506047,5426,-47,47,14.846,1.173601,...,0.0067,15.7204,2.048001,0.0092,14.5591,0.886701,NaN,13.7093,0.036901,NaN
6.08E+18,5139,Mucciarelli,160787,201.727712,-47.505054,5426,-47,47,15.025,1.352601,...,0.0069,16.0554,2.383001,0.0098,14.8595,1.187101,NaN,14.0602,0.387801,0.0075
6.08E+18,5139,Mucciarelli,161739,201.718227,-47.504266,5426,-47,47,15.088,1.415601,...,0.0059,16.0605,2.388101,0.0098,14.8078,1.135401,NaN,14.0464,0.374001,0.0075
6.08E+18,5139,Mucciarelli,162743,201.679732,-47.503383,5426,-47,47,14.800,1.127601,...,0.0077,15.5828,1.910401,0.0089,14.6010,0.928601,NaN,13.8528,0.180401,NaN
6.08E+18,5139,Mucciarelli,165409,201.687549,-47.501240,5426,-47,47,15.189,1.516601,...,0.0058,15.9978,2.325401,0.0074,14.8718,1.199401,0.0064,14.0587,0.386301,NaN


In [7]:
df_full.loc[num_mentions.index[1]]

,NGC,Paper,Author Reference,RAJ2000,DEJ2000,Distance (pc),neg_sigma_dist,pos_sigma_dist,Vmag,Vmag_abs,...,F336W_e,F438W,F438W_abs,F438W_e,F606W,F606W_abs,F606W_e,F814W,F814W_abs,F814W_e
Star ID,,,,,,,,,,,,,,,,,,,,,
R0000101,6121,Carretta,29848,245.892171,-26.551083,1851,-16,15,13.47,2.132968,...,0.0,14.8545,3.517468,0.0,13.0861,1.749068,0.0,11.8955,0.558468,0.0
R0000101,6121,Dorazi + Marino,29848,245.892191,-26.551086,1851,-16,15,NaN,NaN,...,0.0,14.8545,3.517468,0.0,13.0861,1.749068,0.0,11.8955,0.558468,0.0
R0000101,6121,MacLean,13170,245.892167,-26.551083,1851,-16,15,13.52,2.182968,...,0.0,14.8545,3.517468,0.0,13.0861,1.749068,0.0,11.8955,0.558468,0.0
R0000101,6121,Marino,29848,245.892191,-26.551086,1851,-16,15,13.52,2.182968,...,0.0,14.8545,3.517468,0.0,13.0861,1.749068,0.0,11.8955,0.558468,0.0


Weird, the most-repeated one has slightly different values for each entry, but the same source-- almost like the Star ID is actually a cluster identifier and there's 60 stars in the cluster. The other repeat makes more sense because theres a different source each time

In [8]:
df_full.loc[num_mentions.index[2]]

,NGC,Paper,Author Reference,RAJ2000,DEJ2000,Distance (pc),neg_sigma_dist,pos_sigma_dist,Vmag,Vmag_abs,...,F336W_e,F438W,F438W_abs,F438W_e,F606W,F606W_abs,F606W_e,F814W,F814W_abs,F814W_e
Star ID,,,,,,,,,,,,,,,,,,,,,
R0000373,6121,Carretta,30598,245.896600,-26.542350,1851,-16,15,11.999,0.661968,...,0.0,13.6705,2.333468,0.0,11.5831,0.246068,0.0,10.254,-1.083032,0.0
R0000373,6121,Dorazi + Marino,30598,245.896617,-26.542352,1851,-16,15,NaN,NaN,...,0.0,13.6705,2.333468,0.0,11.5831,0.246068,0.0,10.254,-1.083032,0.0
R0000373,6121,MacLean,14037,245.896667,-26.542306,1851,-16,15,12.050,0.712968,...,0.0,13.6705,2.333468,0.0,11.5831,0.246068,0.0,10.254,-1.083032,0.0
R0000373,6121,Marino,30598,245.896617,-26.542352,1851,-16,15,12.050,0.712968,...,0.0,13.6705,2.333468,0.0,11.5831,0.246068,0.0,10.254,-1.083032,0.0


Ok, so all the entries with 4 repeats are from the same 4 sources. All entries with 3 repeats are from 3/4 of these same sources. 

In [9]:
df_full.loc[num_mentions.index[5]]

,NGC,Paper,Author Reference,RAJ2000,DEJ2000,Distance (pc),neg_sigma_dist,pos_sigma_dist,Vmag,Vmag_abs,...,F336W_e,F438W,F438W_abs,F438W_e,F606W,F606W_abs,F606W_e,F814W,F814W_abs,F814W_e
Star ID,,,,,,,,,,,,,,,,,,,,,
R0000781,6121,Dorazi + Marino,31376,245.912191,-26.534324,1851,-16,15,NaN,NaN,...,0.0,14.5838,3.246768,0.0,12.8638,1.526768,0.0,11.6851,0.348068,0.0
R0000781,6121,MacLean,17095,245.912167,-26.534306,1851,-16,15,13.29,1.952968,...,0.0,14.5838,3.246768,0.0,12.8638,1.526768,0.0,11.6851,0.348068,0.0
R0000781,6121,Marino,31376,245.912191,-26.534324,1851,-16,15,13.29,1.952968,...,0.0,14.5838,3.246768,0.0,12.8638,1.526768,0.0,11.6851,0.348068,0.0


Remove repeated data:

In [10]:
isduped=df_full.index.duplicated(keep=False)
df=df_full.iloc[~isduped]
print(df.shape)
print(df.columns)

(752, 41)
Index(['NGC', 'Paper', 'Author Reference', 'RAJ2000', 'DEJ2000',
       'Distance (pc)', 'neg_sigma_dist', 'pos_sigma_dist', 'Vmag', 'Vmag_abs',
       'Bmag', 'Bmag_abs', 'Imag', 'Imag_abs', 'Kmag', 'Kmag_abs', 'Teff',
       'logg', 'age_Kruijssen', 'Fe/H', 'Fe/H_e', 'Fe/H_o', 'O/Fe', 'O/Fe_e',
       'Na/Fe', 'Na/Fe_e', 'F275W', 'F275W_abs', 'F275W_e', 'F336W',
       'F336W_abs', 'F336W_e', 'F438W', 'F438W_abs', 'F438W_e', 'F606W',
       'F606W_abs', 'F606W_e', 'F814W', 'F814W_abs', 'F814W_e'],
      dtype='object')


So with repeats removed, have 752 datapoints. Also 41 columns!

### Check for 99s

Used 99/-99 as NaNs; should replace with actual NaNs now, but let's check:

In [11]:
needed_data=['Fe/H', 'Fe/H_e', 'O/Fe', 'F275W', 'F336W', 'F438W', 'F606W', 'F814W', 
             'F275W_abs', 'F336W_abs', 'F438W_abs', 'F606W_abs', 'F814W_abs','age_Kruijssen']
mins=[]
maxs=[]
for column in needed_data:
    mins.append(df[column].min())
    maxs.append(df[column].max())

if np.min(mins)<-99.99 or np.max(maxs)>99.99:
    print('Likely failed to replace value with NaNs')
else:
    print('Looks good!')

Looks good!


### Look at Data Sizes

In [12]:
y_values=['O/Fe', 'Na/Fe']
filters=['F275W_abs', 'F336W_abs', 'F438W_abs', 'F606W_abs', 'F814W_abs']
df[y_values].describe()

,O/Fe,Na/Fe
count,566.000000,683.000000
mean,0.171186,0.340912
std,0.316409,0.272038
min,-0.913000,-0.420000
25%,0.000000,0.117500
50%,0.217500,0.366000
75%,0.399420,0.530000
max,0.918000,1.384000


In [13]:
df[filters].describe()

,F275W_abs,F336W_abs,F438W_abs,F606W_abs,F814W_abs
count,596.000000,595.000000,581.000000,744.000000,747.000000
mean,4.532625,2.306972,1.370791,-0.074828,-1.095255
std,1.446690,1.347590,1.354845,1.714413,1.743485
min,2.364362,0.377162,-1.161438,-2.886038,-4.195690
25%,3.648809,1.455972,0.393025,-1.246360,-2.286996
50%,4.246253,2.088585,1.146855,-0.253930,-1.286921
75%,5.031687,2.749293,2.218585,0.786917,-0.135991
max,14.220168,12.240468,9.921285,9.645368,8.108690


In [14]:
df_y=df.dropna(subset=y_values)                           
df_y[y_values].describe()

,O/Fe,Na/Fe
count,527.000000,527.000000
mean,0.163545,0.352443
std,0.320878,0.270165
min,-0.913000,-0.420000
25%,-0.010750,0.132000
50%,0.210000,0.377000
75%,0.397682,0.530000
max,0.908415,1.384000


Number of points with both y values

In [15]:
df_filters=df_y.dropna(subset=filters)
df_filters[filters].describe()

,F275W_abs,F336W_abs,F438W_abs,F606W_abs,F814W_abs
count,405.000000,405.000000,405.000000,405.000000,405.000000
mean,4.624964,2.197596,1.137571,-0.538219,-1.583349
std,1.371252,1.158704,1.224340,1.335193,1.408666
min,2.412162,0.377162,-1.161438,-2.883138,-4.195690
25%,3.739579,1.373881,0.166007,-1.537327,-2.624260
50%,4.300150,1.976401,0.914198,-0.694321,-1.754799
75%,5.140297,2.635782,1.860479,0.318383,-0.730317
max,8.512040,6.973785,7.088385,5.669685,4.842485


Number of points with both y values and all filters

In [16]:
other_xs=['age_Kruijssen', 'Fe/H']
df_x_y=df.dropna(subset=y_values+filters+other_xs)
df_x_y[other_xs].describe()

,age_Kruijssen,Fe/H
count,296.000000,296.000000
mean,11.830372,-1.433909
std,0.711953,0.500380
min,10.830000,-2.427000
25%,11.250000,-1.782000
50%,11.630000,-1.460000
75%,12.220000,-1.147750
max,13.060000,-0.398000


Number of points with both y values and all obvously useful x values. QUESTION: WHY DO WE NOT HAVE NORE METALLICITY DATA? OR AGE DATA? THIS SHOULD BE CLUSTER-WIDE PROPERTIES, ASSUMED HAD ALL THESE?

In [17]:
len(df['NGC'].unique())

27

Number of unique clusters

In [18]:
population=df_x_y['NGC'].value_counts()
print(population)

NGC
5139    56
2808    35
6388    32
6093    23
3201    16
6809    14
6254    13
362     13
1261    12
4590    12
288     10
7078    10
6205    10
6838    10
6934     7
6584     6
6121     6
7099     5
104      2
5904     2
6397     2
Name: count, dtype: int64


Cluster population

### How Many Points Have a Single Y-value?

In [19]:
for target in y_values:
    df_y=df.dropna(subset=target)
    print(df_y[target].describe())

count    566.000000
mean       0.171186
std        0.316409
min       -0.913000
25%        0.000000
50%        0.217500
75%        0.399420
max        0.918000
Name: O/Fe, dtype: float64
count    683.000000
mean       0.340912
std        0.272038
min       -0.420000
25%        0.117500
50%        0.366000
75%        0.530000
max        1.384000
Name: Na/Fe, dtype: float64


### How Many Points Have All Useful X-values and a Single Y-value?

Our dataset is not large. Probably need train/test split for each individual target.

In [20]:
for target in y_values:
    subset=filters+other_xs+[target]
    df_x_y=df.dropna(subset=subset)
    print(df_x_y[subset].describe())

        F275W_abs   F336W_abs   F438W_abs   F606W_abs   F814W_abs  \
count  323.000000  323.000000  323.000000  323.000000  323.000000   
mean     4.637354    2.268769    1.275558   -0.362774   -1.391116   
std      1.484420    1.213039    1.239410    1.345266    1.422614   
min      2.364362    0.377162   -1.161438   -2.883138   -4.195690   
25%      3.656722    1.375962    0.347390   -1.313798   -2.396396   
50%      4.296607    2.028545    1.077510   -0.459699   -1.503555   
75%      5.150597    2.840360    2.158141    0.463620   -0.426050   
max      8.512040    6.973785    7.088385    5.669685    4.842485   

       age_Kruijssen        Fe/H        O/Fe  
count     323.000000  323.000000  323.000000  
mean       11.890557   -1.473180    0.185082  
std         0.723695    0.507814    0.290716  
min        10.830000   -2.427000   -0.834000  
25%        11.520000   -1.809000    0.035500  
50%        12.030000   -1.520000    0.220612  
75%        12.220000   -1.153000    0.405000  
ma

# STOP ANALYSIS OF FULL DATASET HERE